# Lab: Perturbing a hybrid AI weather model (NeuralGCM)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/evanwellmeyer/teaching/blob/main/ai_weather_lab/neuralgcm_lab.ipynb)

**NeuralGCM** (Google, Kochkov et al. 2024, *Nature*) is a *hybrid* model: a conventional dynamical core solves the primitive equations on the sphere, and neural networks replace the physics parameterizations (clouds, convection, radiation, boundary layer, ...). The networks were trained on ERA5.

Because the large-scale dynamics are computed explicitly, a perturbation you add has to evolve through real advection, adjustment and wave propagation. In this lab you will:

1. Run a 5-day forecast and compare it with ERA5.
2. Test the **butterfly effect**: add a tiny temperature perturbation and watch the error grow.
3. Inject a **dynamical perturbation** (a vortex, mid-tropospheric subsidence or ascent, a warm or cold anomaly, or drying) on a **start date you choose**, and follow how the atmosphere adjusts and responds.
4. **Compare with your classmates**, who chose different dates and perturbations.

### Before you start
* `Runtime → Change runtime type → T4 GPU` (or any GPU), then `Runtime → Run all` up to Part 1.
* The first forecast takes ~1–2 minutes extra while JAX compiles the model. Each new forecast length or output interval compiles once more; repeat runs are fast.
* Cells with input fields (boxes and drop-down menus) are the ones you are meant to edit. The collapsed cells hold setup and helper code; just run them.

In [ ]:
#@title Install packages (~1 min)
!pip install -q neuralgcm gcsfs cartopy
# If the import cell below fails with a version error, uncomment this line, run it, then
# restart the session (Runtime -> Restart session) and continue from the import cell:
# !pip install -q "neuralgcm==1.2.2" "dinosaur==1.3.6"

In [ ]:
#@title Imports and GPU check
import pickle
import time

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import gcsfs
import jax
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from dinosaur import horizontal_interpolation, spherical_harmonic, xarray_utils
from IPython.display import HTML, display
from matplotlib import animation

import neuralgcm

print("JAX devices:", jax.devices())
if jax.default_backend() != "gpu":
    print("\n*** No GPU found. Use Runtime -> Change runtime type -> GPU, then rerun. ***")

plt.rcParams.update({"font.size": 11, "axes.titlesize": 12, "figure.dpi": 110,
                     "animation.embed_limit": 100})  # MB, for animations shown in the notebook
G = 9.80665       # m s^-2, converts geopotential to geopotential height
A_EARTH = 6.371e6  # m
OMEGA = 7.292e-5   # s^-1
RD = 287.05        # J kg^-1 K^-1

## Setup: load the model and the initial state

Available checkpoints (all trained on ERA5):

| checkpoint | grid spacing | notes |
|---|---|---|
| `v1/deterministic_2_8_deg.pkl` | 2.8° | fastest, recommended for this lab |
| `v1/deterministic_1_4_deg.pkl` | 1.4° | sharper, ~4× slower |
| `v1/stochastic_1_4_deg.pkl` | 1.4° | stochastic physics: each random seed gives a different ensemble member |

The 0.7° model also exists but likely needs more GPU memory than a free-tier T4.

In [ ]:
#@title Choose the model
MODEL_NAME = "v1/deterministic_2_8_deg.pkl"  #@param ["v1/deterministic_2_8_deg.pkl", "v1/deterministic_1_4_deg.pkl", "v1/stochastic_1_4_deg.pkl"]

gcs = gcsfs.GCSFileSystem(token="anon")
with gcs.open(f"gs://neuralgcm/models/{MODEL_NAME}", "rb") as f:
    ckpt = pickle.load(f)
model = neuralgcm.PressureLevelModel.from_checkpoint(ckpt)

print("Prognostic inputs:", model.input_variables)
print("Forcings (held fixed during the forecast):", model.forcing_variables)

The initial state comes from **ERA5** (Google's public ARCO-ERA5 copy on Cloud Storage, no account needed). It is on a 0.25° grid with 37 pressure levels, so we average it conservatively onto the model grid.

The default start date, **8 Feb 2021**, is just before the February 2021 cold-air outbreak over Texas, a large-amplitude midlatitude wave pattern. Any date from 1940 to 2025 works (00 UTC is safest).

In [ ]:
#@title Start date, then load ERA5 (~1–2 min)
START_TIME = "2021-02-08T00:00"  #@param {type:"string"}
T0 = np.datetime64(START_TIME)

ERA5_PATH = "gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3"
era5 = xr.open_zarr(ERA5_PATH, chunks=None, storage_options=dict(token="anon"))

era5_grid = spherical_harmonic.Grid(
    latitude_nodes=era5.sizes["latitude"],
    longitude_nodes=era5.sizes["longitude"],
    latitude_spacing=xarray_utils.infer_latitude_spacing(era5.latitude),
    longitude_offset=xarray_utils.infer_longitude_offset(era5.longitude),
)
regridder = horizontal_interpolation.ConservativeRegridder(
    era5_grid, model.data_coords.horizontal, skipna=True
)


def to_model_grid(ds):
    return xarray_utils.fill_nan_with_nearest(xarray_utils.regrid(ds, regridder))


def load_ic(start_time):
    # ERA5 initial condition (model inputs + forcings) at `start_time`, on the model grid
    raw = (
        era5[model.input_variables + model.forcing_variables]
        # The model expects forcings (SST, sea ice) lagged by 24 h, as in training
        .pipe(xarray_utils.selective_temporal_shift,
              variables=model.forcing_variables, time_shift="24 hours")
        .sel(time=[np.datetime64(start_time)])
        .compute()
    )
    return to_model_grid(raw)


tic = time.time()
IC = load_ic(START_TIME)  # initial condition on the model grid
print(f"Loaded and regridded in {time.time() - tic:.0f} s")
IC

In [ ]:
#@title Helper functions: running the model, diagnostics, maps
def run_forecast(ic, days, output_hours=24, seed=0, levels=None):
    # Run NeuralGCM from the dataset `ic` (one time) for `days` days.
    # Forcings (SST, sea ice) are held at their initial values (persistence).
    # Returns an xarray Dataset with dimension `lead_day` (0 = initial state).
    # `levels` (hPa) keeps only those pressure levels, to save memory.
    n_out = days * 24 // output_hours + 1
    inputs = model.inputs_from_xarray(ic.isel(time=0))
    forcings = model.forcings_from_xarray(ic.isel(time=0))
    state = model.encode(inputs, forcings, jax.random.key(seed))
    _, preds = model.unroll(
        state,
        model.forcings_from_xarray(ic.head(time=1)),
        steps=n_out,
        timedelta=np.timedelta64(output_hours, "h"),
        start_with_input=True,
    )
    lead_hours = np.arange(n_out) * output_hours
    out = model.data_to_xarray(preds, times=lead_hours)
    out = out.map(lambda da: da.copy(data=np.asarray(da.data)))  # JAX arrays -> NumPy, so xarray tools work
    if levels is not None:
        out = out.sel(level=list(levels))
    return out.rename(time="lead_day").assign_coords(lead_day=lead_hours / 24)


def era5_truth(days, variables=("geopotential", "temperature"), levels=(500, 850)):
    # ERA5 at the forecast's daily valid times, regridded to the model grid.
    times = T0 + np.arange(days + 1) * np.timedelta64(1, "D")
    ds = era5[list(variables)].sel(time=times, level=list(levels)).compute()
    ds = to_model_grid(ds)
    return ds.rename(time="lead_day").assign_coords(lead_day=np.arange(days + 1))


def z500(ds):
    # 500 hPa geopotential height (m)
    return ds["geopotential"].sel(level=500) / G


def area_rms(da):
    # cos(latitude)-weighted RMS over the globe
    w = np.cos(np.deg2rad(da.latitude))
    return np.sqrt((da ** 2).weighted(w).mean(("latitude", "longitude")))


def gaussian_blob(lat0, lon0, radius_km, grid=None):
    # exp(-(d/R)^2), d = great-circle distance from (lat0, lon0)
    grid = IC if grid is None else grid
    lat, lon = np.deg2rad(grid.latitude), np.deg2rad(grid.longitude)
    la0, lo0 = np.deg2rad(lat0), np.deg2rad(lon0)
    cosd = np.sin(lat) * np.sin(la0) + np.cos(lat) * np.cos(la0) * np.cos(lon - lo0)
    dist_km = 6371.0 * np.arccos(np.clip(cosd, -1.0, 1.0))
    return np.exp(-((dist_km / radius_km) ** 2))


def add_to(ds, var, anomaly):
    # Return a copy of ds with `anomaly` (broadcastable DataArray) added to `var`
    out = ds.copy(deep=True)
    out[var] = (ds[var] + anomaly).transpose(*ds[var].dims)
    return out


def _cyclic(da):
    # Repeat the first longitude at +360 so maps have no seam
    first = da.isel(longitude=[0])
    return xr.concat([da, first.assign_coords(longitude=first.longitude + 360)], "longitude")


def plot_maps(fields, titles, cmap="RdBu_r", vmax=None, contours=None, cbar_label="",
              ncols=2, central_longitude=180, region=None, separate_scales=False):
    # Panel of maps. `contours` (optional) is a matching list of fields drawn as black lines.
    # region = (lat0, lon0, half_width_lon, half_width_lat) zooms in on a box around (lat0, lon0).
    # separate_scales=True gives every panel its own color scale and color bar.
    n = len(fields)
    nrows = int(np.ceil(n / ncols))
    if region is None:
        proj = ccrs.Robinson(central_longitude)
        figsize = (6.5 * ncols, 3.4 * nrows)
    else:
        lat0, lon0, hw_lon, hw_lat = region
        proj = ccrs.PlateCarree(central_longitude=lon0)
        figsize = (5.2 * ncols, 5.2 * nrows * hw_lat / hw_lon + 0.8 * nrows)
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize, squeeze=False,
                             subplot_kw={"projection": proj})
    diverging = cmap in ("RdBu_r", "BrBG", "PuOr_r", "coolwarm")

    def limits(fs):
        top = float(vmax) if vmax is not None else float(max(np.abs(f).max() for f in fs)) or 1.0
        return (-top, top) if diverging else (float(min(f.min() for f in fs)), top)

    lo, hi = limits(fields)
    for i, ax in enumerate(axes.flat):
        if i >= n:
            ax.remove()
            continue
        if separate_scales:
            lo, hi = limits([fields[i]])
        f = _cyclic(fields[i]).transpose("latitude", "longitude")  # pcolormesh/contour need (lat, lon)
        mesh = ax.pcolormesh(f.longitude, f.latitude, f, transform=ccrs.PlateCarree(),
                             cmap=cmap, vmin=lo, vmax=hi, shading="auto")
        if contours is not None:
            c = _cyclic(contours[i]).transpose("latitude", "longitude")
            ax.contour(c.longitude, c.latitude, c, levels=12, colors="k", linewidths=0.6,
                       transform=ccrs.PlateCarree())
        ax.add_feature(cfeature.COASTLINE, linewidth=0.4, edgecolor="0.3")
        if region is None:
            ax.set_global()
        else:
            ax.set_extent([-hw_lon, hw_lon, max(lat0 - hw_lat, -89.0), min(lat0 + hw_lat, 89.0)],
                          crs=proj)
        ax.set_title(titles[i])
        if separate_scales:
            fig.colorbar(mesh, ax=ax, orientation="horizontal", shrink=0.8, pad=0.06)
    if separate_scales:
        fig.suptitle(cbar_label)
    else:
        fig.colorbar(mesh, ax=axes, orientation="horizontal", shrink=0.5, pad=0.04, label=cbar_label)
    plt.show()


def animate_maps(field, title, contours=None, cbar_label="", central_longitude=180, region=None,
                 rescale=False, interval_ms=300):
    # Animated map of a difference `field` with dimension lead_day, one frame per lead time, with play controls.
    # `title(t)` gives the title for lead day t. `contours` (optional, same lead days) are drawn as black lines.
    # region is as in plot_maps; rescale=True gives every frame its own color scale.
    if region is None:
        proj = ccrs.Robinson(central_longitude)
        figsize = (8, 4.8)
    else:
        lat0, lon0, hw_lon, hw_lat = region
        proj = ccrs.PlateCarree(central_longitude=lon0)
        figsize = (8, 8 * hw_lat / hw_lon + 1.3)
    f = _cyclic(field).transpose("lead_day", "latitude", "longitude")
    top = float(np.abs(f).max()) or 1.0
    fig, ax = plt.subplots(figsize=figsize, subplot_kw={"projection": proj}, layout="constrained")
    mesh = ax.pcolormesh(f.longitude, f.latitude, f.isel(lead_day=0), transform=ccrs.PlateCarree(),
                         cmap="RdBu_r", vmin=-top, vmax=top, shading="auto")
    ax.add_feature(cfeature.COASTLINE, linewidth=0.4, edgecolor="0.3")
    if region is None:
        ax.set_global()
    else:
        ax.set_extent([-hw_lon, hw_lon, max(lat0 - hw_lat, -89.0), min(lat0 + hw_lat, 89.0)], crs=proj)
    fig.colorbar(mesh, ax=ax, orientation="horizontal", shrink=0.7, label=cbar_label)
    if contours is not None:
        c = _cyclic(contours).transpose("lead_day", "latitude", "longitude")
        levels = np.linspace(float(c.min()), float(c.max()), 14)[1:-1]  # same levels in every frame
    lines = []

    def draw(i):
        frame = f.isel(lead_day=i)
        mesh.set_array(frame.values)
        if rescale:
            t = float(np.abs(frame).max()) or 1.0
            mesh.set_clim(-t, t)
        if contours is not None:
            for cs in lines:
                cs.remove()
            lines[:] = [ax.contour(c.longitude, c.latitude, c.isel(lead_day=i), levels=levels, colors="k",
                                   linewidths=0.6, transform=ccrs.PlateCarree())]
        ax.set_title(title(float(f.lead_day[i])))

    draw(0)
    anim = animation.FuncAnimation(fig, draw, frames=f.sizes["lead_day"], interval=interval_ms)
    display(HTML(anim.to_jshtml(default_mode="once")))
    plt.close(fig)


In [ ]:
#@title Perturbation toolkit: winds from a streamfunction or velocity potential, balance, diagnostics
# ---- horizontal derivatives on the sphere (work on any field with latitude/longitude) ----
def _coslat(da):
    return np.cos(np.deg2rad(da.latitude))


def d_dlon(da):
    # d/d(longitude) per radian, periodic in longitude
    dlon = np.deg2rad(float(da.longitude[1] - da.longitude[0]))
    return (da.roll(longitude=-1, roll_coords=False) - da.roll(longitude=1, roll_coords=False)) / (2 * dlon)


def d_dlat(da):
    # d/d(latitude) per radian (centered differences, fine on the model's Gaussian grid)
    lat = da.latitude.values
    return da.assign_coords(latitude=np.deg2rad(lat)).differentiate("latitude").assign_coords(latitude=lat)


def winds_from_streamfunction(psi):
    # Non-divergent (rotational) wind: u = -(1/a) dpsi/dphi, v = (1/(a cos phi)) dpsi/dlambda
    return -d_dlat(psi) / A_EARTH, d_dlon(psi) / (A_EARTH * _coslat(psi))


def winds_from_velocity_potential(chi):
    # Irrotational (divergent) wind: u = (1/(a cos phi)) dchi/dlambda, v = (1/a) dchi/dphi
    return d_dlon(chi) / (A_EARTH * _coslat(chi)), d_dlat(chi) / A_EARTH


def relative_vorticity(u, v):
    c = _coslat(u)
    return (d_dlon(v) - d_dlat(u * c)) / (A_EARTH * c)


def divergence(u, v):
    c = _coslat(u)
    return (d_dlon(u) + d_dlat(v * c)) / (A_EARTH * c)


# ---- vertical structure and balance ----
def layer_profile(level, top_hPa, bottom_hPa, shape):
    # 'bump':   sin^2(pi x): 1 in the middle, zero with zero slope at the layer edges
    # 'dipole': -sin(2 pi x): negative in the upper half, positive in the lower half,
    #           = d(bump)/dx / pi, so it integrates to zero over the layer
    x = (level - top_hPa) / (bottom_hPa - top_hPa)
    prof = np.sin(np.pi * x) ** 2 if shape == "bump" else -np.sin(2 * np.pi * x)
    return prof.where((x >= 0) & (x <= 1), 0.0)


def hydrostatic_geopotential(dT):
    # Geopotential anomaly in hydrostatic balance with dT, zero at the lowest level:
    # dphi(p) = R_d * integral of dT dln(p), from p down to the bottom level
    Tr = dT.sortby("level", ascending=False)
    lnp = np.log(Tr.level)
    layer = 0.5 * (Tr + Tr.shift(level=1)) * (lnp.shift(level=1) - lnp)
    return (RD * layer.fillna(0.0).cumsum("level")).sortby("level")


def hydrostatic_temperature(dphi):
    # Temperature anomaly in hydrostatic balance with dphi: T' = -(p / R_d) dphi'/dp
    p = dphi.level * 100.0
    dphi_dp = dphi.assign_coords(level=p.values).differentiate("level").assign_coords(level=dphi.level.values)
    return -(p / RD) * dphi_dp


def vertical_velocity(div):
    # Pressure velocity from mass continuity, omega(p) = -integral of D dp from the model top
    # (Pa/s, positive = descent)
    Dr = div.sortby("level")
    p = Dr.level * 100.0
    layer = 0.5 * (Dr + Dr.shift(level=1)) * (p - p.shift(level=1))
    return -layer.fillna(0.0).cumsum("level")


# ---- the perturbations ----
PERTURBATIONS = {
    # name: (typical amplitude, what AMPLITUDE means)
    "cyclonic vortex": (2.0, "peak relative vorticity in 1e-5 s^-1, same sign as f"),
    "anticyclonic vortex": (2.0, "peak relative vorticity in 1e-5 s^-1, opposite sign to f"),
    "subsidence": (0.3, "peak descent at mid-layer, omega in Pa/s"),
    "ascent": (0.3, "peak ascent at mid-layer, omega in Pa/s"),
    "warm anomaly": (3.0, "peak temperature change in K"),
    "cold anomaly": (3.0, "peak temperature change in K"),
    "drying": (0.5, "fraction of the specific humidity removed at the center (0-1)"),
}
OPPOSITE = {"cyclonic vortex": "anticyclonic vortex", "anticyclonic vortex": "cyclonic vortex",
            "subsidence": "ascent", "ascent": "subsidence",
            "warm anomaly": "cold anomaly", "cold anomaly": "warm anomaly"}


def _at(da, lat0, lon0):
    # Value(s) at the grid point nearest (lat0, lon0)
    return da.sel(latitude=lat0, longitude=lon0 % 360, method="nearest")


def make_perturbation(ic, kind, amplitude, lat0, lon0, radius_km, top_hPa, bottom_hPa, balanced=True):
    # Build the anomaly fields for one perturbation. Returns {variable name: anomaly DataArray}.
    if kind not in PERTURBATIONS:
        raise ValueError(f"Unknown perturbation {kind!r}. Choose from {list(PERTURBATIONS)}")
    if not top_hPa < bottom_hPa:
        raise ValueError("LAYER_TOP_HPA must be a smaller pressure (higher up) than LAYER_BOTTOM_HPA")
    if amplitude == 0:
        amplitude = PERTURBATIONS[kind][0]
    grid_km = 20000.0 / ic.sizes["latitude"]
    if radius_km < 2.5 * grid_km:
        print(f"Warning: a {radius_km:.0f} km radius is small for this grid (~{grid_km:.0f} km spacing); "
              "the model will smooth it out.")

    # Center on the nearest grid point, so the imposed peak equals the amplitude exactly
    lat0 = float(ic.latitude.sel(latitude=lat0, method="nearest"))
    lon0 = float(ic.longitude.sel(longitude=lon0 % 360, method="nearest"))
    g = gaussian_blob(lat0, lon0, radius_km, grid=ic)  # horizontal shape, 1 at the center
    bump = layer_profile(ic.level, top_hPa, bottom_hPa, "bump")
    anoms = {}

    if kind in ("warm anomaly", "cold anomaly"):
        sign = 1.0 if kind == "warm anomaly" else -1.0
        anoms["temperature"] = sign * amplitude * g * bump
        if balanced:  # raise/lower the heights above, as the hypsometric equation requires
            anoms["geopotential"] = hydrostatic_geopotential(anoms["temperature"])

    elif kind in ("cyclonic vortex", "anticyclonic vortex"):
        hemisphere = 1.0 if lat0 >= 0 else -1.0
        zeta0 = (1.0 if kind == "cyclonic vortex" else -1.0) * hemisphere * amplitude * 1e-5
        u1, v1 = winds_from_streamfunction(g)  # unit streamfunction, used to set the scale
        psi = zeta0 / float(_at(relative_vorticity(u1, v1), lat0, lon0)) * g * bump
        anoms["u_component_of_wind"], anoms["v_component_of_wind"] = winds_from_streamfunction(psi)
        if balanced and abs(lat0) >= 10:
            f0 = 2 * OMEGA * np.sin(np.deg2rad(lat0))
            anoms["geopotential"] = f0 * psi  # geostrophic (linear) balance with f fixed at the center
            anoms["temperature"] = hydrostatic_temperature(anoms["geopotential"])
            big = float(np.abs(anoms["temperature"]).max())
            if big > 10:
                print(f"Warning: the balanced temperature anomaly reaches {big:.0f} K, stronger than most "
                      "real vortices. Consider a smaller AMPLITUDE, a larger radius or a deeper layer.")
        elif balanced:
            print("Note: within 10 degrees of the equator f is too small for geostrophic balance, "
                  "so only the winds are changed.")

    elif kind in ("subsidence", "ascent"):
        omega0 = (1.0 if kind == "subsidence" else -1.0) * amplitude
        # omega = omega0 sin^2(pi x) in the layer  =>  D = -d(omega)/dp: convergence above, divergence below
        dp = (bottom_hPa - top_hPa) * 100.0
        div_center = (omega0 * np.pi / dp) * layer_profile(ic.level, top_hPa, bottom_hPa, "dipole")
        u1, v1 = winds_from_velocity_potential(g)  # unit velocity potential, used to set the scale
        chi = div_center / float(_at(divergence(u1, v1), lat0, lon0)) * g
        anoms["u_component_of_wind"], anoms["v_component_of_wind"] = winds_from_velocity_potential(chi)

    elif kind == "drying":
        q = ic["specific_humidity"].isel(time=0, drop=True)
        anoms["specific_humidity"] = -min(amplitude, 1.0) * q * g * bump

    return anoms


def apply_perturbation(ic, anoms):
    for var, anomaly in anoms.items():
        ic = add_to(ic, var, anomaly)
    return ic


def describe_perturbation(anoms, kind, lat0, lon0, radius_km):
    # Print the size of the perturbation and plot a map plus vertical profiles at its center
    prof = {}
    if "u_component_of_wind" in anoms:
        u, v = anoms["u_component_of_wind"], anoms["v_component_of_wind"]
        print(f"Largest wind change: {float(np.sqrt(u ** 2 + v ** 2).max()):.1f} m/s")
        if "vortex" in kind:
            prof["vorticity (1e-5 s^-1)"] = relative_vorticity(u, v) * 1e5
        else:
            div = divergence(u, v)
            prof["divergence (1e-5 s^-1)"] = div * 1e5
            prof["omega (Pa/s, + = down)"] = vertical_velocity(div)
    if "temperature" in anoms:
        prof["temperature (K)"] = anoms["temperature"]
    if "geopotential" in anoms:
        prof["height (m)"] = anoms["geopotential"] / G
    if "specific_humidity" in anoms:
        prof["specific humidity (g/kg)"] = anoms["specific_humidity"] * 1000

    name0, field0 = next(iter(prof.items()))
    column = _at(field0, lat0, lon0)
    peak_level = int(column.level[int(np.abs(column).argmax())])
    for name, f in prof.items():
        c = _at(f, lat0, lon0)
        print(f"  {name:28s} peak at center: {float(c[int(np.abs(c).argmax())]):+.3g} "
              f"at {int(c.level[int(np.abs(c).argmax())])} hPa")

    n = len(prof)
    fig = plt.figure(figsize=(5.5 + 2.6 * n, 3.8))
    hw_lat = max(12.0, 3.5 * radius_km / 111.0)
    proj = ccrs.PlateCarree(central_longitude=lon0)
    ax = fig.add_subplot(1, n + 2, (1, 2), projection=proj)
    m = _cyclic(field0.sel(level=peak_level)).transpose("latitude", "longitude")
    top = float(np.abs(m).max()) or 1.0
    mesh = ax.pcolormesh(m.longitude, m.latitude, m, transform=ccrs.PlateCarree(),
                         cmap="RdBu_r", vmin=-top, vmax=top, shading="auto")
    if "u_component_of_wind" in anoms:
        uu = anoms["u_component_of_wind"].sel(level=peak_level).transpose("latitude", "longitude")
        vv = anoms["v_component_of_wind"].sel(level=peak_level).transpose("latitude", "longitude")
        # about 18 arrows across the box; the longest arrow is 1/12 of the panel width
        dlon = float(uu.longitude[1] - uu.longitude[0])
        step = max(1, int(round(3.2 * hw_lat / 18 / dlon)))
        near = (np.abs(((uu.longitude - lon0 + 180) % 360) - 180) <= 1.6 * hw_lat) & (np.abs(uu.latitude - lat0) <= hw_lat)
        uu, vv = uu.where(near), vv.where(near)
        uu, vv = uu.isel(latitude=slice(None, None, step), longitude=slice(None, None, step)), \
                 vv.isel(latitude=slice(None, None, step), longitude=slice(None, None, step))
        speed = float(np.sqrt(uu ** 2 + vv ** 2).max()) or 1.0
        ax.quiver(uu.longitude.values, uu.latitude.values, uu.values, vv.values, transform=ccrs.PlateCarree(),
                  scale=12 * speed, scale_units="width", color="0.15", width=0.0035)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.4, edgecolor="0.3")
    ax.set_extent([-1.6 * hw_lat, 1.6 * hw_lat, max(lat0 - hw_lat, -89.0), min(lat0 + hw_lat, 89.0)], crs=proj)
    ax.set_title(f"{name0}, {peak_level} hPa")
    fig.colorbar(mesh, ax=ax, orientation="horizontal", shrink=0.8, pad=0.08)
    for k, (name, f) in enumerate(prof.items()):
        pax = fig.add_subplot(1, n + 2, k + 3)
        c = _at(f, lat0, lon0)
        pax.plot(c, c.level, "o-", lw=2, ms=3)
        pax.axvline(0, color="0.6", lw=0.8)
        pax.set_ylim(1000, 50)
        pax.set_title(name, fontsize=10)
        if k == 0:
            pax.set_ylabel("Pressure (hPa)")
    fig.suptitle(f"Imposed {kind} at ({lat0:g}°, {lon0:g}°E)")
    fig.tight_layout()
    plt.show()


# ---- response diagnostics ----
def _lev(da, level):
    return da if level is None else da.sel(level=level)


FIELDS = {
    # name: (function of (forecast dataset, level or None), units)
    "relative vorticity": (lambda ds, L: relative_vorticity(_lev(ds.u_component_of_wind, L),
                                                            _lev(ds.v_component_of_wind, L)) * 1e5, "1e-5 s^-1"),
    "divergence": (lambda ds, L: divergence(_lev(ds.u_component_of_wind, L),
                                            _lev(ds.v_component_of_wind, L)) * 1e5, "1e-5 s^-1"),
    "geopotential height": (lambda ds, L: _lev(ds.geopotential, L) / G, "m"),
    "temperature": (lambda ds, L: _lev(ds.temperature, L), "K"),
    "zonal wind": (lambda ds, L: _lev(ds.u_component_of_wind, L), "m/s"),
    "meridional wind": (lambda ds, L: _lev(ds.v_component_of_wind, L), "m/s"),
    "specific humidity": (lambda ds, L: _lev(ds.specific_humidity, L) * 1000, "g/kg"),
}


def field(ds, name, level=None):
    return FIELDS[name][0](ds, level)


def response(exp, name, level=None, run="perturbed"):
    # Difference between a run in `exp` and the control, for one field
    return field(exp[run], name, level) - field(exp["control"], name, level)

---
## Part 1: Control forecast vs. ERA5

Run a 5-day forecast from the ERA5 state and compare 500 hPa heights with what actually happened.

In [ ]:
#@title Run the control forecast
FORECAST_DAYS = 5  #@param {type:"integer"}

tic = time.time()
control = run_forecast(IC, FORECAST_DAYS)
truth = era5_truth(FORECAST_DAYS)
print(f"Forecast done in {time.time() - tic:.0f} s (includes JIT compilation on the first run)")

days_to_show = sorted({d for d in (1, 3, 5, FORECAST_DAYS) if d <= FORECAST_DAYS})
fields, titles = [], []
for d in days_to_show:
    fields += [z500(truth).sel(lead_day=d), z500(control).sel(lead_day=d)]
    titles += [f"ERA5, day {d}", f"NeuralGCM, day {d}"]
plot_maps(fields, titles, cmap="viridis", vmax=float(z500(truth).max()),
          cbar_label="500 hPa geopotential height (m)")

In [ ]:
#@title Forecast error vs. lead time
err = area_rms(z500(control) - z500(truth))
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(err.lead_day, err, "o-", lw=2)
ax.set(xlabel="Lead time (days)", ylabel="Global RMS error (m)",
       title="Z500 error of the NeuralGCM forecast vs. ERA5")
ax.grid(alpha=0.3)
plt.show()

---
## Part 2: The butterfly effect

Add random noise with a standard deviation of only `NOISE_K` kelvin to the temperature at every grid point and level, run again, and measure how far the perturbed forecast drifts from the control.

In a chaotic atmosphere, tiny differences should grow roughly exponentially and then saturate when the two forecasts are as different as two random weather states.

In [ ]:
#@title Perturbed forecast
NOISE_K = 0.01  #@param {type:"number"}
BUTTERFLY_DAYS = 7  #@param {type:"integer"}

rng = np.random.default_rng(1)
noise = xr.DataArray(rng.normal(0.0, NOISE_K, IC["temperature"].shape), dims=IC["temperature"].dims)
IC_noisy = add_to(IC, "temperature", noise)

control_long = run_forecast(IC, BUTTERFLY_DAYS)
noisy = run_forecast(IC_noisy, BUTTERFLY_DAYS)
diff = z500(noisy) - z500(control_long)

# A rough "saturation" level: RMS difference between two unrelated states (day 0 vs. last day)
saturation = float(area_rms(z500(control_long).isel(lead_day=-1) - z500(control_long).isel(lead_day=0)))

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.semilogy(diff.lead_day, area_rms(diff), "o-", lw=2, label=f"perturbed - control ({NOISE_K} K noise)")
ax.axhline(saturation, color="0.5", ls="--", label="~ difference between unrelated states")
ax.set(xlabel="Lead time (days)", ylabel="Global RMS Z500 difference (m)",
       title="Growth of a tiny initial perturbation")
ax.legend(fontsize=9)
ax.grid(alpha=0.3, which="both")
plt.show()

show = [d for d in (1, 3, 5, 7) if d <= BUTTERFLY_DAYS]
plot_maps([diff.sel(lead_day=d) for d in show], [f"Z500 difference, day {d}" for d in show],
          contours=[z500(control_long).sel(lead_day=d) for d in show],
          cbar_label="Perturbed - control Z500 (m); contours: control Z500")

---
## Part 3: A dynamical perturbation

Now you choose a **start date** and a **perturbation**, inject it into the ERA5 initial state, and compare a perturbed forecast with an unperturbed control. Each student should pick a different date (and season, or location), so as a class you sample many different background flows.

The options for `PERTURBATION`, with what each one changes in the initial state and what `AMPLITUDE` means:

* **`cyclonic vortex`, `anticyclonic vortex`**: winds from a streamfunction $\psi$, so they are non-divergent: $u = -\partial\psi/\partial y$, $v = \partial\psi/\partial x$.
  * `AMPLITUDE`: peak relative vorticity in $10^{-5}\ \mathrm{s^{-1}}$ (typical value 2).
* **`subsidence`, `ascent`**: winds from a velocity potential $\chi$, so they are irrotational: $u = \partial\chi/\partial x$, $v = \partial\chi/\partial y$. Convergence above mid-layer and divergence below means descent, through mass continuity $\omega(p) = -\int_0^p D\,dp'$. The profile is $\omega = \omega_0 \sin^2(\pi x)$ across the layer.
  * `AMPLITUDE`: peak $\omega$ at mid-layer in Pa/s (typical value 0.3).
* **`warm anomaly`, `cold anomaly`**: temperature.
  * `AMPLITUDE`: peak $\Delta T$ in K (typical value 3).
* **`drying`**: specific humidity.
  * `AMPLITUDE`: fraction removed at the center (typical value 0.5).

Every perturbation has a Gaussian shape of radius `RADIUS_KM` around (`CENTER_LAT`, `CENTER_LON`), and a smooth vertical profile between `LAYER_TOP_HPA` and `LAYER_BOTTOM_HPA` (peaking mid-layer). Set `AMPLITUDE = 0` to use the typical value.

**`BALANCED`** adds a mass field consistent with the perturbation:
* for a vortex, a geopotential anomaly in geostrophic balance, $\phi' = f_0\psi'$, and the temperature anomaly that keeps it hydrostatic, $T' = -(p/R_d)\,\partial\phi'/\partial p$;
* for a warm or cold anomaly, the hydrostatic change in geopotential above it.

A balanced cyclonic vortex peaking mid-layer is a positive potential-vorticity anomaly: it comes with cold air below its center and warm air above, as in PV-inversion pictures.

Without balance, the model has to **adjust** the initial state itself. For subsidence or ascent the winds are deliberately unbalanced, which is the point: how does the atmosphere respond to imposed vertical motion?

Three forecasts are run: the **control**, your **perturbed** forecast, and a **butterfly** forecast with only 0.01 K of random temperature noise (from Part 2). The butterfly run shows how large a difference chaos alone produces, so you can tell whether your response stands out from it.

In [ ]:
#@title Design your perturbation
EXPERIMENT_DATE = "2021-02-08T00:00"  #@param {type:"string"}
PERTURBATION = "cyclonic vortex"  #@param ["cyclonic vortex", "anticyclonic vortex", "subsidence", "ascent", "warm anomaly", "cold anomaly", "drying"]
AMPLITUDE = 0.0  #@param {type:"number"}
CENTER_LAT = 45.0  #@param {type:"number"}
CENTER_LON = 180.0  #@param {type:"number"}
RADIUS_KM = 1000.0  #@param {type:"number"}
LAYER_TOP_HPA = 250  #@param {type:"integer"}
LAYER_BOTTOM_HPA = 850  #@param {type:"integer"}
BALANCED = True  #@param {type:"boolean"}

if EXPERIMENT_DATE == START_TIME:
    IC_EXP, _ic_exp_date = IC, START_TIME
elif globals().get("_ic_exp_date") != EXPERIMENT_DATE:
    print(f"Loading ERA5 for {EXPERIMENT_DATE} (~1-2 min)...")
    IC_EXP, _ic_exp_date = load_ic(EXPERIMENT_DATE), EXPERIMENT_DATE

AMP = AMPLITUDE if AMPLITUDE != 0 else PERTURBATIONS[PERTURBATION][0]
print(f"{PERTURBATION}, amplitude {AMP:g} ({PERTURBATIONS[PERTURBATION][1]})")
ANOMS = make_perturbation(IC_EXP, PERTURBATION, AMP, CENTER_LAT, CENTER_LON, RADIUS_KM,
                          LAYER_TOP_HPA, LAYER_BOTTOM_HPA, BALANCED)
describe_perturbation(ANOMS, PERTURBATION, CENTER_LAT, CENTER_LON, RADIUS_KM)

In [ ]:
#@title Run the control, perturbed and butterfly forecasts
EXPERIMENT_DAYS = 10  #@param {type:"integer"}
OUTPUT_HOURS = 6  #@param [3, 6, 12] {type:"raw"}
KEEP_LEVELS = [200, 250, 300, 400, 500, 600, 700, 850, 925, 1000]  # saved to save memory

tic = time.time()
rng = np.random.default_rng(1)
tiny = xr.DataArray(rng.normal(0.0, 0.01, IC_EXP["temperature"].shape), dims=IC_EXP["temperature"].dims)
kwargs = dict(output_hours=int(OUTPUT_HOURS), levels=KEEP_LEVELS)
EXP = {
    "setup": dict(date=EXPERIMENT_DATE, kind=PERTURBATION, amp=AMP, lat=CENTER_LAT, lon=CENTER_LON,
                  radius=RADIUS_KM, top=LAYER_TOP_HPA, bottom=LAYER_BOTTOM_HPA, balanced=BALANCED,
                  days=EXPERIMENT_DAYS, kwargs=kwargs),
    "ic": IC_EXP,
    "control": run_forecast(IC_EXP, EXPERIMENT_DAYS, **kwargs),
}
print(f"control done ({time.time() - tic:.0f} s)")
EXP["perturbed"] = run_forecast(apply_perturbation(IC_EXP, ANOMS), EXPERIMENT_DAYS, **kwargs)
print(f"perturbed done ({time.time() - tic:.0f} s)")
EXP["butterfly"] = run_forecast(add_to(IC_EXP, "temperature", tiny), EXPERIMENT_DAYS, **kwargs)
print(f"butterfly done ({time.time() - tic:.0f} s)")

In [ ]:
#@title Animation of the response
MAP_FIELD = "relative vorticity"  #@param ["relative vorticity", "divergence", "geopotential height", "temperature", "zonal wind", "meridional wind", "specific humidity"]
MAP_LEVEL_HPA = 500  #@param [200, 250, 300, 400, 500, 600, 700, 850, 925, 1000] {type:"raw"}
FRAME_HOURS = 6  #@param [6, 12, 24] {type:"raw"}
MAP_VIEW = "regional"  #@param ["regional", "global"]
RESCALE_EACH_FRAME = False  #@param {type:"boolean"}
# The regional box is centered on your perturbation; shift it east to follow a wave train downstream
SHIFT_EAST_DEG = 0  #@param {type:"number"}
SHIFT_NORTH_DEG = 0  #@param {type:"number"}
HALF_WIDTH_LON_DEG = 70  #@param {type:"number"}
HALF_WIDTH_LAT_DEG = 35  #@param {type:"number"}

s = EXP["setup"]
level = int(MAP_LEVEL_HPA)
region = (s["lat"] + SHIFT_NORTH_DEG, s["lon"] + SHIFT_EAST_DEG, HALF_WIDTH_LON_DEG, HALF_WIDTH_LAT_DEG)
step = max(1, int(FRAME_HOURS) // s["kwargs"]["output_hours"])
d = response(EXP, MAP_FIELD, level).isel(lead_day=slice(None, None, step))
heights = field(EXP["control"], "geopotential height", level).isel(lead_day=slice(None, None, step))
print("Building the animation...")
animate_maps(d, lambda t: f"{MAP_FIELD}, {level} hPa, day {t:g}", contours=heights,
             cbar_label=f"Perturbed - control {MAP_FIELD} ({FIELDS[MAP_FIELD][1]}); "
                        f"contours: control {level} hPa height",
             region=region if MAP_VIEW == "regional" else None, rescale=RESCALE_EACH_FRAME)


In [ ]:
#@title Animated vertical cross-section along the perturbation's latitude
SECTION_FIELD = "temperature"  #@param ["temperature", "relative vorticity", "divergence", "geopotential height", "zonal wind", "meridional wind", "specific humidity"]
FRAME_HOURS = 6  #@param [6, 12, 24] {type:"raw"}
SECTION_HALF_WIDTH_DEG = 60  #@param {type:"integer"}
RESCALE_EACH_FRAME = False  #@param {type:"boolean"}

s = EXP["setup"]
step = max(1, int(FRAME_HOURS) // s["kwargs"]["output_hours"])
lat_sec = float(EXP["control"].latitude.sel(latitude=s["lat"], method="nearest"))
sec = response(EXP, SECTION_FIELD).sel(latitude=lat_sec).isel(lead_day=slice(None, None, step))
sec = sec.assign_coords(dlon=((sec.longitude - s["lon"] + 180) % 360) - 180)
sec = sec.swap_dims({"longitude": "dlon"}).sortby("dlon")
sec = sec.sel(dlon=slice(-SECTION_HALF_WIDTH_DEG, SECTION_HALF_WIDTH_DEG)).transpose("lead_day", "level", "dlon")
top = float(np.abs(sec).max()) or 1.0

fig, ax = plt.subplots(figsize=(7, 4.2), layout="constrained")
mesh = ax.pcolormesh(sec.dlon, sec.level, sec.isel(lead_day=0), cmap="RdBu_r", vmin=-top, vmax=top, shading="auto")
ax.invert_yaxis()
ax.set(xlabel="Degrees longitude from the center", ylabel="Pressure (hPa)")
fig.colorbar(mesh, ax=ax, label=f"Perturbed - control {SECTION_FIELD} ({FIELDS[SECTION_FIELD][1]})")


def draw(i):
    frame = sec.isel(lead_day=i)
    mesh.set_array(frame.values)
    if RESCALE_EACH_FRAME:
        t = float(np.abs(frame).max()) or 1.0
        mesh.set_clim(-t, t)
    ax.set_title(f"Perturbed - control along {lat_sec:.1f}°, day {float(sec.lead_day[i]):g}")


draw(0)
anim = animation.FuncAnimation(fig, draw, frames=sec.sizes["lead_day"], interval=300)
display(HTML(anim.to_jshtml(default_mode="once")))
plt.close(fig)


**Is the response bigger than chaos?** The solid line is the size of the change in 500 hPa height caused by your perturbation, as a root-mean-square (RMS) over the whole globe. The dashed line is the same for the butterfly run, which only added 0.01 K of random noise. Once the two lines meet, the difference your perturbation makes can no longer be told apart from chaos. The vertical axis is logarithmic: each grid line is a factor of 10.

In [ ]:
#@title Is the response bigger than chaos?
sig = area_rms(response(EXP, "geopotential height", 500))
noise = area_rms(response(EXP, "geopotential height", 500, run="butterfly"))

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.semilogy(sig.lead_day, sig.where(sig > 0), lw=2, label=f"your perturbation ({EXP['setup']['kind']})")
ax.semilogy(noise.lead_day, noise.where(noise > 0), lw=2, ls="--", color="0.4", label="0.01 K noise (chaos)")
ax.set(title="Global RMS change in 500 hPa height", xlabel="Lead time (days)", ylabel="RMS change (m), log scale")
ax.legend()
ax.grid(alpha=0.3, which="both")
plt.show()


**Questions** (answer those that fit your perturbation)

*Vortex*
1. Compare the imposed vorticity with the vorticity at 6 h and day 1. How much of it survives the first day? Run once with `BALANCED = False` and look at `divergence` at 6 h: what do you see spreading out, and how fast?
2. The Rossby radius of deformation is $L_R = NH/f \approx 1000$ km in midlatitudes. With `BALANCED = False`, try `RADIUS_KM = 400` and `2500`. Which vortex keeps more of its vorticity, and which ends up mostly as a height anomaly? Explain with geostrophic adjustment and potential-vorticity conservation.
3. Where does the vortex go? Compare its track with the background flow (the contours). Does a cyclone and an anticyclone drift the same way? (Think about the $\beta$ effect.)
4. After a few days, is the signal still near the center or is it in a wave train downstream? Estimate how fast that wave train spreads compared with the jet.

*Subsidence or ascent*
1. Use the cross-section of `temperature`. What happens in the first 6–24 h where air is descending? Estimate the expected adiabatic warming from $\partial T/\partial t \approx \omega\,(\kappa T/p - \partial T/\partial p)$ and compare.
2. Look at `divergence` maps at 0.25 and 1 day. Most of the imposed divergence disappears quickly. Where does it go?
3. From the vorticity equation, $\partial\zeta/\partial t \approx -f D$. Predict the sign of the vorticity change above and below mid-layer, then check with the `relative vorticity` cross-section.
4. Is there a lasting response after day 2? What kind of balanced state is left behind?

*Warm/cold anomaly or drying*
1. With `BALANCED = True`, is the anomaly advected passively, or does it set up circulations of its own? Compare with `BALANCED = False`.
2. For drying: does removing moisture change the dynamics, or only the moisture field? Why might the answer depend on whether the region is about to have convection or a frontal system?

*Everyone*
* When does your perturbation's RMS change in Z500 fall below, or merge with, the butterfly curve? After that time, can you tell the "real" response apart from chaos?

In [ ]:
#@title (Optional) Is the response linear? Run the opposite perturbation
# The regional box is centered on your perturbation; shift it east to follow a wave train downstream
SHIFT_EAST_DEG = 0  #@param {type:"number"}
SHIFT_NORTH_DEG = 0  #@param {type:"number"}
HALF_WIDTH_LON_DEG = 70  #@param {type:"number"}
HALF_WIDTH_LAT_DEG = 35  #@param {type:"number"}

s = EXP["setup"]
region = (s["lat"] + SHIFT_NORTH_DEG, s["lon"] + SHIFT_EAST_DEG, HALF_WIDTH_LON_DEG, HALF_WIDTH_LAT_DEG)
if s["kind"] not in OPPOSITE:
    print(f"No opposite perturbation is defined for {s['kind']}.")
else:
    opp = make_perturbation(EXP["ic"], OPPOSITE[s["kind"]], s["amp"], s["lat"], s["lon"], s["radius"],
                            s["top"], s["bottom"], s["balanced"])
    if "opposite" not in EXP:  # run once per experiment, so changing the map box is quick
        EXP["opposite"] = run_forecast(apply_perturbation(EXP["ic"], opp), s["days"], **s["kwargs"])
    plus = response(EXP, "geopotential height", 500)
    minus = response(EXP, "geopotential height", 500, run="opposite")
    nonlinear = 0.5 * (plus + minus)  # zero if the response just changes sign
    ratio = area_rms(nonlinear) / area_rms(plus)

    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.plot(ratio.lead_day, ratio, lw=2)
    ax.set(xlabel="Lead time (days)", ylabel="RMS(nonlinear part) / RMS(response)",
           title=f"{s['kind']} vs. {OPPOSITE[s['kind']]}: Z500", ylim=(0, None))
    ax.grid(alpha=0.3)
    plt.show()

    show = [t for t in (1, 3, 6) if t <= s["days"]]
    plot_maps([plus.sel(lead_day=t) for t in show] + [nonlinear.sel(lead_day=t) for t in show],
              [f"response, day {t:g}" for t in show] + [f"nonlinear part, day {t:g}" for t in show],
              cbar_label="Z500 (m)", ncols=len(show), region=region)

If the atmosphere responded linearly, the opposite perturbation would give exactly the opposite response and the "nonlinear part" would be zero. When does it stop being small, and why? (Remember that chaos itself is a nonlinear process.)

---
## Part 4: Compare with your classmates

Share your setup (date, perturbation, amplitude, location, and whether it was balanced) with the class, together with your two most informative figures. Then, as a class or in groups:

1. **Same perturbation, different dates.** How much does the response depend on the background flow, for example the jet's position and strength, or the season? Which dates gave the largest downstream response, and what did those days have in common?
2. **Same date, different perturbations.** Compare a cyclonic and an anticyclonic vortex, or subsidence and ascent, from the same date. Are the responses mirror images?
3. **Balanced vs. unbalanced.** Did adding a balanced mass field change the response after day 1, or only during the first hours?
4. **Signal vs. chaos.** For how many of the class's experiments did the response stay above the butterfly curve until day 5? What does that say about how many runs you would need to make a robust claim about, say, a remote response?
5. Is NeuralGCM's behavior physically sensible in every case? Point to any experiment where it looks wrong, and suggest why (for example, an initial state far from anything in the training data).